# Covariance, dependence and tails

Family 5 connects sample moments, factor covariance, joint dependence and extreme tails. Mature libraries do the numerical work; Convexity supplies explicit mathematical contracts and composition with the completed probability and transform families.

This problem book uses only generated data. Every section includes an identity or a recovery check. [Finite catalog and contracts](../docs/dependence.md) · [Executable script](../examples/dependence.py).

Install the project with the `notebooks` extra and run all cells. No broker or external data is needed.

In [ ]:
from pathlib import Path
import sys
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "convexity" / "math").is_dir())
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from numpy.testing import assert_allclose
from convexity.math import dependence as dep, distributions as dist, transforms as tx

rng = np.random.default_rng(42)
plt.rcParams.update({"figure.figsize": (9, 3.6), "axes.grid": True, "grid.alpha": .18})

## 1. Sample covariance and shrinkage

Rows are observations; columns are coordinates. With estimated location, sample covariance is
$S=(X-\bar X)^T(X-\bar X)/(n-\mathrm{ddof})$.

Shrinkage uses the population divisor and a spherical target: $(1-\alpha)S+\alpha\,\mathrm{tr}(S)I/d$.
Ledoit–Wolf and OAS estimate the shrinkage intensity. Scaling the coordinates changes that target.

In [ ]:
loadings = np.array([[1., .2], [.5, 1.], [1.5, -.4], [-.2, .8], [.7, .7], [1., -.6]])
factors = rng.normal(size=(3000, 2))
observations = factors @ loadings.T + .4 * rng.normal(size=(3000, 6))
truth = dep.FactorCovariance(loadings, np.eye(2), np.full(6, .4**2))
sample = dep.sample_covariance(observations)
assert_allclose(sample.covariance, np.cov(observations, rowvar=False))

small = observations[:18]
fits = {"Sample": dep.sample_covariance(small, ddof=0),
        "Ledoit–Wolf": dep.LedoitWolf().fit(small),
        "OAS": dep.OAS().fit(small),
        "Fixed 20%": dep.ShrunkCovariance(.2).fit(small)}
pd.DataFrame({name: {"shrinkage": fit.shrinkage,
                     "smallest eigenvalue": np.linalg.eigvalsh(fit.covariance)[0],
                     "distance to generating covariance": np.linalg.norm(fit.covariance-truth.covariance)}
              for name, fit in fits.items()}).T

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(11, 3.3), layout="constrained")
for ax, (name, matrix) in zip(axes, [("Generating covariance", truth.covariance),
                                   ("18-observation sample", fits["Sample"].covariance),
                                   ("Ledoit–Wolf estimate", fits["Ledoit–Wolf"].covariance)]):
    im = ax.imshow(matrix, cmap="RdBu_r", vmin=-2, vmax=2)
    ax.set(title=name, xlabel="coordinate", ylabel="coordinate")
    ax.grid(False)
fig.colorbar(im, ax=axes, shrink=.8, label="covariance")
plt.show()

## 2. EWMA is a weighted finite-history estimate

The newest row receives the largest weight. We normalize $w_i\propto\lambda^{n-1-i}$ and expose the effective count $1/\sum w_i^2$. With `ddof=1`, centered covariance is divided by $1-\sum w_i^2$.

The count measures weight concentration; it does not correct for serial dependence. At decay 1, this agrees with ordinary sample covariance.

In [ ]:
decay = .97
weighted = dep.ewma_covariance(observations, decay=decay, ddof=1)
weights = decay ** np.arange(len(observations)-1, -1, -1.)
weights /= weights.sum()
assert_allclose(weighted.location, weights @ observations)
assert_allclose(weighted.covariance, np.cov(observations, rowvar=False, aweights=weights, ddof=1))
assert_allclose(dep.ewma_covariance(observations, decay=1., ddof=1).covariance, sample.covariance)
print("Rows:", weighted.nobs, "Effective observations:", weighted.effective_nobs)

linear_weights = np.ones(6)/6
assert_allclose(sample.variance(linear_weights), np.var(observations @ linear_weights, ddof=1))

## 3. Factor geometry composes with covariance

For $X=\mu+B F+\epsilon$, uncorrelated factors and residuals imply
$\operatorname{Cov}(X)=B\operatorname{Cov}(F)B^T+\operatorname{Cov}(\epsilon)$.

OLS exposures to observed factors recover this decomposition exactly when full residual covariance is retained. A diagonal residual covariance imposes an additional assumption. Family-4 latent factor fits can supply their loadings and noise directly.

In [ ]:
observed_factors = dep.fit_factor_covariance(observations, factors, residual="full")
assert_allclose(observed_factors.covariance, sample.covariance, atol=1e-12)
assert_allclose(observed_factors.loadings, loadings, atol=.03)

latent = tx.FactorAnalysis(2).fit(observations)
composed = dep.FactorCovariance(latent.loadings, np.eye(2), latent.noise_variance, location=latent.location)
assert_allclose(composed.covariance, latent.covariance)
pd.DataFrame({"generating noise": np.full(6, .16), "fitted latent noise": latent.noise_variance})

## 4. Correlation and PSD constraints

Pearson, Spearman and Kendall tau-b are different dependence summaries. A covariance matrix must be PSD, but it can be singular. A precision matrix needs an invertible covariance.

Spectral clipping minimizes Frobenius distance subject to an eigenvalue floor. Nearest-correlation projection also holds the diagonal at one. Repairs are explicit results with a distance and convergence report.

In [ ]:
correlations = {method: dep.correlation(observations, method=method)[0, 1]
                for method in ("pearson", "spearman", "kendall")}
print(correlations)
assert_allclose(dep.correlation_to_covariance(sample.correlation, sample.standard_deviation), sample.covariance)

invalid = np.array([[1., 1.2], [1.2, 1.]])
psd = dep.PositiveSemidefinite(min_eigenvalue=1e-6).project(invalid)
unit = dep.nearest_correlation(invalid, min_eigenvalue=1e-6)
assert dep.PositiveSemidefinite().contains(psd.matrix)
assert_allclose(np.diag(unit.matrix), 1.)
pd.DataFrame({"input": np.linalg.eigvalsh(invalid), "PSD projection": np.linalg.eigvalsh(psd.matrix),
              "unit-diagonal projection": np.linalg.eigvalsh(unit.matrix)}, index=["minimum", "maximum"])

## 5. Student-t shape is not covariance

An elliptical Student-t has a shared radial scale; its coordinates can remain dependent even with diagonal shape. For $\nu>2$, covariance is $\nu/(\nu-2)$ times shape. Below that threshold there is no finite covariance.

The likelihood fit uses SciPy optimization with Cholesky shape coordinates. Degrees of freedom can be fixed or estimated within explicit bounds. The fitted law is the completed family-2 distribution, including conditioning.

In [ ]:
correlation = np.array([[1., .6], [.6, 1.]])
heavy = dep.multivariate_student_t(6, [0., 1.], correlation)
draws = heavy.sample(6000, rng=rng)
fitted_t = dep.StudentT(df=5., estimate_df=True).fit(draws)
assert_allclose(fitted_t.location, [0., 1.], atol=.08)
assert_allclose(fitted_t.covariance, fitted_t.shape*fitted_t.df/(fitted_t.df-2))
conditional = fitted_t.distribution.condition([0], [1.])
print("Fitted df:", fitted_t.df)
print("Shape diagonal:", fitted_t.shape.diagonal())
print("Covariance diagonal:", fitted_t.covariance.diagonal())
print("Conditional mean of coordinate 1 given coordinate 0=1:", conditional.mean())

## 6. Same rank dependence, different joint tails

All four copulas below have population Kendall tau 0.5. Their tail behavior differs. Gaussian tail dependence vanishes asymptotically, Student-t dependence occurs in both tails, Clayton concentrates in the lower tail and Gumbel in the upper tail.

These are uniform coordinates. Marginal scale and shape are supplied separately.

In [ ]:
rho = np.sin(np.pi/4)
rank_correlation = [[1., rho], [rho, 1.]]
copulas = {"Gaussian": dep.GaussianCopula(rank_correlation),
           "Student-t, df=4": dep.StudentTCopula(rank_correlation, 4),
           "Clayton, theta=2": dep.ClaytonCopula(2),
           "Gumbel, theta=2": dep.GumbelCopula(2)}
fig, axes = plt.subplots(1, 4, figsize=(12, 3), layout="constrained")
for ax, (name, copula) in zip(axes, copulas.items()):
    uniforms = copula.sample(1800, rng=rng)
    ax.scatter(*uniforms.T, s=3, alpha=.25)
    ax.set(title=name, xlabel="uniform coordinate 1", xlim=(0, 1), ylim=(0, 1))
    ax.set_aspect("equal")
axes[0].set_ylabel("uniform coordinate 2")
plt.show()
pd.DataFrame({name: copula.tail_dependence() for name, copula in copulas.items()},
             index=["lower tail coefficient", "upper tail coefficient"]).T

## 7. Sklar composition and rank estimation

Combining a t copula with t marginals of the same degrees of freedom recovers the multivariate t law exactly. Combining it with other continuous marginals changes marginal distributions while preserving the copula.

Kendall fitting is invariant under increasing marginal transformations. Its reported likelihood is evaluated at average ranks divided by $n+1$, and is a pseudo-likelihood. A t-copula rank fit needs supplied df; tau does not identify it.

In [ ]:
copula = dep.StudentTCopula(correlation, 6)
joint = dep.CopulaDistribution(copula, [dist.student_t(6), dist.student_t(6, loc=1)])
assert_allclose(joint.log_density(draws[:100]), heavy.log_density(draws[:100]), atol=1e-10)
rank_fit = dep.fit_copula(draws, family="student_t", df=6)
assert isinstance(rank_fit.copula, dep.StudentTCopula)
assert_allclose(rank_fit.copula.correlation, correlation, atol=.04)

mixed = dep.CopulaDistribution(copula, [dist.lognormal(log_scale=.4), dist.normal(2, 3)])
mixed_samples = mixed.sample(1000, rng=rng)
assert np.all(mixed_samples[:, 0] > 0)
print("Fitted latent correlation:", rank_fit.copula.correlation[0, 1])
print("Joint CDF at marginal medians:", copula.cdf([.5, .5], rng=rng))
print("Pseudo log likelihood:", rank_fit.log_likelihood)

## 8. Peaks over threshold: conditional and original-sample probabilities

We fit GPD excesses above a supplied threshold. If $p_u$ is the observed exceedance rate, then for $x\ge u$,
$P(X>x)=p_u\,\overline G(x-u)$.

The return level at period $T$ has survival $1/T$ in **original observation units**, not excess-event units. The model defines only the upper tail. Threshold selection, clustering and parameter uncertainty are not inferred automatically.

In [ ]:
threshold = 3.
excess_law = dep.generalized_pareto(.2, scale=1.5)
excesses = excess_law.sample(4000, rng=rng)
tail_data = np.r_[np.zeros(16000), threshold + excesses]
pot = dep.peaks_over_threshold(tail_data, threshold)
periods = np.geomspace(5, 10000, 120)
levels = pot.return_level(periods)
assert_allclose(pot.survival(levels), 1/periods, rtol=1e-12)
assert_allclose(pot.exceedance_rate, .2)
print("xi, scale, exceedance rate:", pot.tail.xi, pot.tail.scale, pot.exceedance_rate)
pd.DataFrame({"period (observations)": [10, 100, 1000], "return level": pot.return_level([10, 100, 1000])})

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), layout="constrained")
grid = np.linspace(0, np.quantile(excesses, .995), 250)
axes[0].semilogy(grid, excess_law.survival(grid), label="generating excess law")
axes[0].semilogy(grid, pot.tail.distribution.survival(grid), "--", label="fitted GPD")
axes[0].set(xlabel="excess above threshold", ylabel="conditional survival", title="Excess-tail recovery")
axes[0].legend()
generating_levels = threshold + 1.5/.2 * ((.2*periods)**.2 - 1)
axes[1].semilogx(periods, generating_levels, label="generating tail")
axes[1].semilogx(periods, levels, "--", label="fitted POT")
axes[1].set(xlabel="period (original observations)", ylabel="return level", title="Extrapolation beyond the threshold")
axes[1].legend()
plt.show()

## 9. Block maxima and GEV shape sign

GEV return periods are in blocks. `block_maxima` uses explicit, nonoverlapping positional blocks and rejects an incomplete final block by default.

Both tail models expose conventional extreme-value index $\xi$: positive means a heavy upper tail and negative means a finite upper endpoint. SciPy's `genextreme` uses the opposite sign; the adapter handles that mapping.

In [ ]:
series = rng.exponential(size=24000)
maxima = dep.block_maxima(series, block_size=24)
gev = dep.GeneralizedExtremeValue().fit(maxima)
return_periods = np.array([10., 100., 1000.])
assert_allclose(gev.distribution.survival(gev.return_level(return_periods)), 1/return_periods)
print("GEV xi:", gev.xi, "(exponential maxima approach Gumbel, xi=0)")
pd.DataFrame({"period (blocks)": return_periods, "return level": gev.return_level(return_periods)})

## 10. User-written tools use the same contracts

A covariance estimator implements `.fit(samples)` and returns inspectable covariance state. Copulas can instead supply the public adapter protocol; continuous marginal objects need CDF, quantile and log-density methods. Neither extension requires registration.

In [ ]:
class DiagonalEstimator:
    def fit(self, samples):
        return dep.CovarianceEstimate(samples.mean(axis=0),
                                      np.diag(samples.var(axis=0, ddof=1)), nobs=len(samples))

custom = dep.fit_covariance(observations, DiagonalEstimator())
assert_allclose(np.diag(custom.covariance), np.diag(sample.covariance))
assert not custom.covariance.flags.writeable
print("All covariance, dependence and tail identities passed.")